<a href="https://colab.research.google.com/github/gpruthvi177-design/New/blob/master/Financial_News_Summarization_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import json
import pandas as pd

# ==============================================================================
# MODULE 1: DATASET INGESTION & CLEANING (PANDAS & REGEX)
# ==============================================================================

# 1. Create a flat CSV file locally in Colab (simulating flat-file storage, no DBMS)
sample_news = [
    {
        "article_id": "NEWS-101",
        "source": "Economic Times",
        "category": "Markets",
        "raw_text": """<p>Indian benchmark indices Sensex and Nifty surged to record highs on Thursday!
        Foreign institutional investors pumped over Rs 4,500 crore into banking and IT stocks...
        Visit https://economictimes.com/markets for live updates. Analysts noted that strong quarterly
        earnings growth and stable domestic inflation figures boosted overall market sentiment.</p>""",
        "reference_summary": "Sensex and Nifty hit record highs driven by strong FII inflows into banking and IT stocks amid solid earnings and stable inflation."
    },
    {
        "article_id": "NEWS-102",
        "source": "Mint",
        "category": "Energy & Economy",
        "raw_text": """<div>Global crude oil prices jumped 4.2% following fresh supply disruption concerns
        in key maritime shipping corridors!! &amp; Aviation and paint manufacturing stocks faced sharp
        selling pressure on Dalal Street. Economists warned that sustained energy price volatility
        could delay anticipated interest rate cuts by central banks in the near term.</div>""",
        "reference_summary": "Surging global crude oil prices triggered sell-offs in aviation and paint stocks due to margin pressures and potential delays in rate cuts."
    },
    {
        "article_id": "NEWS-103",
        "source": "Moneycontrol",
        "category": "Banking & Policy",
        "raw_text": """<article>The Reserve Bank of India (RBI) kept the benchmark repo rate unchanged at 6.5%
        for the consecutive policy review. Bond yields remained steady while real estate and automobile
        stocks traded flat following the announcement. Market participants are now awaiting upcoming
        industrial production data for clearer cues on domestic demand momentum.</article>""",
        "reference_summary": "The RBI held the repo rate steady at 6.5 percent with a neutral stance, leaving bond yields and auto stocks largely unchanged."
    }
]

# Save to flat CSV file and ingest back via Pandas (as specified in Module 1)
pd.DataFrame(sample_news).to_csv("financial_news_dataset.csv", index=False)
df = pd.read_csv("financial_news_dataset.csv")

# 2. Text Normalization & Noise Removal using Python Regex
def clean_and_normalize(text):
    text = re.sub(r'<[^>]+>', ' ', str(text))        # Remove HTML tags (<p>, <div>, etc.)
    text = re.sub(r'http\S+|www\.\S+', '', text)     # Remove URLs/links
    text = re.sub(r'&[a-z]+;', ' ', text)            # Remove HTML entities (&amp;, etc.)
    text = re.sub(r'[^a-zA-Z0-9.,%()\s-]', '', text) # Remove noisy special artifacts
    text = re.sub(r'\s+', ' ', text).strip()         # Normalize whitespace
    return text

# 3. Sentence Segmentation using Python Regex
def segment_sentences(clean_text):
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', clean_text) if len(s.strip()) > 10]
    return sentences

# Execute Module 1 Pipeline
df['cleaned_text'] = df['raw_text'].apply(clean_and_normalize)
df['segmented_sentences'] = df['cleaned_text'].apply(segment_sentences)
df['sentence_count'] = df['segmented_sentences'].apply(len)

print("Module 1 Complete: Flat CSV Ingested, Noise Removed, and Sentences Segmented.\n")
display(df[['article_id', 'source', 'sentence_count', 'cleaned_text']])

print("\nSample Segmented Output (Article NEWS-101):")
for i, sent in enumerate(df.loc[0, 'segmented_sentences'], 1):
    print(f"  Sentence {i}: {sent}")

Module 1 Complete: Flat CSV Ingested, Noise Removed, and Sentences Segmented.



,article_id,source,sentence_count,cleaned_text
0,NEWS-101,Economic Times,3,Indian benchmark indices Sensex and Nifty surg...
1,NEWS-102,Mint,2,Global crude oil prices jumped 4.2% following ...
2,NEWS-103,Moneycontrol,3,The Reserve Bank of India (RBI) kept the bench...



Sample Segmented Output (Article NEWS-101):
  Sentence 1: Indian benchmark indices Sensex and Nifty surged to record highs on Thursday Foreign institutional investors pumped over Rs 4,500 crore into banking and IT stocks...
  Sentence 2: Visit for live updates.
  Sentence 3: Analysts noted that strong quarterly earnings growth and stable domestic inflation figures boosted overall market sentiment.
